In [0]:
# src/notebooks/01_data_profiling.py
import pandas as pd
from pyspark.sql.functions import (
    col, count, when, isnull, trim, length, countDistinct, 
    min, max, avg, round as spark_round, desc
)

# 1. CONFIGURATION
CATALOG = "vstone_catalog"
LANDING_PATH = f"/Volumes/{CATALOG}/raw/landing"

FILES = {
    "1_main": {"path": f"{LANDING_PATH}/1_main.csv", "format": "csv", "opts": {"sep": ","}},
    "catalogs": {"path": f"{LANDING_PATH}/catalogs.csv", "format": "csv", "opts": {"sep": ";"}},
    "geolocation": {"path": f"{LANDING_PATH}/final_geografic.csv", "format": "csv", "opts": {"sep": ","}},
    "text": {"path": f"{LANDING_PATH}/1_text.csv", "format": "csv", "opts": {"sep": ",", "multiLine": "true", "escape": '"'}},
    "photos": {"path": f"{LANDING_PATH}/1_photo.csv", "format": "csv", "opts": {"sep": ","}},
}

def profile_file(name, path, fmt, options):
    print(f"\n{'='*85}\n📄 EXECUTIVE SUMMARY: {name.upper()}\n{'='*85}")
    
    # Load Data
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("encoding", "UTF-8")
          .options(**options)
          .format(fmt)
          .load(path))
    
    total_rows = df.count()
    if total_rows == 0: return None, 0

    # --- 1. VERTICAL QUALITY AUDIT ---
    null_exprs = [count(when(isnull(c) | (trim(col(c)) == ""), c)).alias(c) for c in df.columns]
    pdf_stats = df.select(null_exprs).toPandas().transpose()
    pdf_stats.columns = ["Incomplete_Count"]
    
    pdf_stats["Incomplete_Pct"] = ((pdf_stats["Incomplete_Count"].astype(float) / total_rows) * 100).round(2)
    pdf_stats["Data_Type"] = [dict(df.dtypes)[c] for c in pdf_stats.index]
    
    print(f"📊 Dataset Size: {total_rows:,} rows | {len(df.columns)} columns")
    display(pdf_stats.style.background_gradient(cmap='Reds', subset=['Incomplete_Pct']))

    # --- 2. SMART DATA INTEGRITY CHECK (Composite Key Logic) ---
    # Agar catalog hai toh use combination of columns for uniqueness
    if name == "catalogs":
        pk_candidate = "Composite (Марка+Модель+Поколение+Комплектация)"
        distinct_count = df.select("Марка", "Модель", "Поколение", "Комплектация").distinct().count()
    else:
        pk_candidate = "id" if "id" in df.columns else df.columns[0]
        distinct_count = df.select(pk_candidate).distinct().count()
    
    integrity_data = pd.DataFrame([
        {"Metric": "Primary Key Candidate", "Result": pk_candidate},
        {"Metric": "Uniqueness Status", "Result": "✅ 100% Unique" if distinct_count == total_rows else f"❌ {total_rows-distinct_count:,} Duplicates"},
        {"Metric": "Schema Health", "Result": "✅ Standard" if not [c for c in df.columns if " " in c] else "⚠️ Non-Standard (Cyrillic/Spaces detected)"}
    ])
    display(integrity_data)
    
    return df, total_rows

# --- EXECUTION LOOP ---
profiled = {}
for name, cfg in FILES.items():
    try:
        df, rows = profile_file(name, cfg["path"], cfg["format"], cfg["opts"])
        profiled[name] = {"df": df}
    except Exception as e:
        print(f"❌ Error profiling {name}: {str(e)}")

# --- 3. ADVANCED BUSINESS LOGIC (1_MAIN) ---
if "1_main" in profiled:
    print("\n" + "⭐" * 25 + " 1_MAIN BUSINESS DEEP-DIVE " + "⭐" * 25)
    df_main = profiled["1_main"]["df"]
    
    # KPIs as Vertical Table
    kpis = df_main.select(
        spark_round(avg("cost"), 0).alias("Average Listing Price"),
        min("year").alias("Oldest Model Year"),
        max("year").alias("Newest Model Year"),
        countDistinct("marka").alias("Total Unique Brands")
    ).toPandas().transpose()

    kpis.columns = ["Value"]
    kpis["Value"] = kpis["Value"].astype(str)
    display(kpis)